# 00 - Ekstraksi Data DBLP

Mengambil publikasi conference & workshop (`inproceedings`) tahun 2015-2025 dari dump XML DBLP, lalu disimpan ke `data/raw/dblp_2015_2025_raw.parquet` untuk dipakai di `01_preprocessing.ipynb`.

- Dump harian: https://dblp.org/xml/dblp.xml.gz (sekitar 1,1 GB) beserta `dblp.dtd`. Snapshot bulanan yang tidak berubah ada di https://dblp.org/xml/release/.
- Data yang dipakai di notebook 01-03 adalah snapshot 13-14 November 2026 (ada di zip, lihat `data/README.md`). Dump harian terus diperbarui, jadi kalau diekstrak ulang jumlah record-nya bisa sedikit berbeda.
- `distributionofpublicationtype.csv` diunduh manual dari halaman statistik DBLP: https://dblp.org/statistics/distributionofpublicationtype.html

In [ ]:
import gzip
import os
import urllib.request

import pandas as pd
from lxml import etree

DBLP_DIR = '../data/dblp'
OUT_PATH = '../data/raw/dblp_2015_2025_raw.parquet'
YEAR_MIN, YEAR_MAX = 2015, 2025

## Download dump

In [ ]:
os.makedirs(DBLP_DIR, exist_ok=True)

for name in ['dblp.xml.gz', 'dblp.dtd']:
    path = f'{DBLP_DIR}/{name}'
    if os.path.exists(path):
        print("sudah ada:", path)
        continue
    print("download:", name)
    urllib.request.urlretrieve(f'https://dblp.org/xml/{name}', path)

## Parsing XML

File-nya dibaca bertahap pakai `iterparse`, jadi tidak perlu dimuat sekaligus ke memori, dan setiap record yang sudah diproses langsung dibuang. DTD harus di-load dengan `resolve_entities=True` karena DBLP menulis karakter khusus sebagai entity (misalnya `&uuml;` untuk ü).

| Kolom | Diambil dari |
|---|---|
| `key`, `mdate` | atribut record |
| `authors` | semua elemen `<author>` |
| `title` | `<title>`, termasuk teks di dalam tag seperti `<i>` |
| `pages`, `year`, `crossref`, `url` | elemen dengan nama yang sama |
| `venue` | `<booktitle>` |
| `doi` | `<ee>` pertama yang diawali `https://doi.org/` |

In [ ]:
RECORDS = ('article', 'inproceedings', 'proceedings', 'book', 'incollection', 'phdthesis', 'mastersthesis', 'www', 'data')
COLUMNS = ['key', 'mdate', 'authors', 'title', 'pages', 'year', 'venue', 'doi', 'crossref', 'url']


def parse_inproceedings(path, year_min=2015, year_max=2025):
    rows = []
    with gzip.open(path, 'rb') as f:
        for _, el in etree.iterparse(f, events=('end',), tag=RECORDS, load_dtd=True, resolve_entities=True, huge_tree=True):
            year = el.findtext('year')
            if el.tag == 'inproceedings' and year and year_min <= int(year) <= year_max:
                title = el.find('title')
                ees = [e.text for e in el.findall('ee') if e.text]
                rows.append({
                    'key': el.get('key'),
                    'mdate': el.get('mdate'),
                    'authors': [a.text for a in el.findall('author')],
                    'title': ''.join(title.itertext()) if title is not None else None,
                    'pages': el.findtext('pages'),
                    'year': int(year),
                    'venue': el.findtext('booktitle'),
                    'doi': next((e for e in ees if e.startswith('https://doi.org/')), None),
                    'crossref': el.findtext('crossref'),
                    'url': el.findtext('url'),
                })
            # buang elemen yang sudah diproses biar memori tidak penuh
            el.clear()
            while el.getprevious() is not None:
                del el.getparent()[0]
    return pd.DataFrame(rows, columns=COLUMNS)

In [ ]:
df = parse_inproceedings(f'{DBLP_DIR}/dblp.xml.gz', YEAR_MIN, YEAR_MAX)

print("Jumlah record:", f"{len(df):,}")
print("mdate terbaru:", df['mdate'].max())
df['year'].value_counts().sort_index()

## Simpan

In [ ]:
# parquet yang sudah ada tidak ditimpa, supaya hasil analisis tetap sama.
# hapus dulu file lamanya kalau memang mau ganti snapshot.
if os.path.exists(OUT_PATH):
    print("sudah ada, tidak ditimpa:", OUT_PATH)
else:
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    df.to_parquet(OUT_PATH, index=False)
    print("disimpan:", OUT_PATH)